# Data Cleaning

## Basic Cleaning
Here I'm just cleaning up the data to be better formatted to work with. This is more focused on separating columns with multiple values into multiple columns not just a single one, renaming columns where I feel necessary, dropping some columns, fixing the date time format, and anything else I deem necessary.

### Needed Libraries

In [1]:
import pandas as pd
import numpy as np
import os

### Global Variable

In [65]:
month_choices = [9, 10, 11, 12, 1, 2]

Chaning the working directory so I don't have to hard code long path names

In [66]:
print(f'Current: {os.getcwd()}')
os.chdir('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/htmlparse')
print(f'Current: {os.getcwd()}')

Current: /Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/cleaned
Current: /Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/htmlparse


### Team Statistics

In [67]:
team_stats = pd.read_csv('team_stats.csv')

team_stats = team_stats.drop(columns='Unnamed: 0')
team_stats = team_stats.rename(columns={'Cmp-Att-Yd-TD-INT':'ManyStats'})
team_stats[['Completion', 'Attempts', 'Yards', 'Touchdowns', 'Interceptions']] = team_stats.ManyStats.str.split('-', expand=True)
team_stats = team_stats.drop(columns='ManyStats')

team_stats.columns = team_stats.columns.str.replace(' ', '_')
team_stats.columns = team_stats.columns.str.replace('.', '_')
team_stats.columns = team_stats.columns.str.replace('-', '_')

team_stats[['Success4thDown', 'Failed4thDown']] = team_stats.Fourth_Down_Conv_.str.split('-', expand=True)
team_stats[['Fumbles', 'FumblesLost']] = team_stats.Fumbles_Lost.str.split('-', expand=True)
team_stats[['Penalties', 'PenaltyYards']] = team_stats.Penalties_Yards.str.split('-', expand=True)
#team_stats[['RushAttempts', 'RushYards', 'RushTDs']] = team_stats.Rush_Yds_TDs.str.split('-', expand=True)
team_stats[['Sacked', 'SackYards']] = team_stats.Sacked_Yards.str.split('-', expand=True)
team_stats[['Success3rdDown', 'Failed3rdDown']] = team_stats.Third_Down_Conv_.str.split('-', expand=True)
team_stats[['TimeMinutes', 'TimeSeconds']] = team_stats.Time_of_Possession.str.split(':', expand=True)

team_stats = team_stats.drop(columns={'Fourth_Down_Conv_', 'Fumbles_Lost', 'Penalties_Yards', 'Sacked_Yards', 'Third_Down_Conv_', 'Time_of_Possession'})

team_stats['Rush_Yds_TDs'].str.count('-').value_counts()

team_stats.loc[2083]

team_stats.at[2083,'Rush_Yds_TDs'] = '8-neg1-0'

team_stats[['RushAttempts', 'RushYards', 'RushTDs']] = team_stats.Rush_Yds_TDs.str.split('-', expand=True)
team_stats = team_stats.drop(columns='Rush_Yds_TDs')

team_stats.at[2083, "RushYards"] = -1

skip_cols = ['team_name', 'date']
cols_to_change = [col for col in team_stats.columns if col not in skip_cols]
team_stats[cols_to_change] = team_stats[cols_to_change].astype(int)

team_stats['CompletionPct'] = round(100 * (team_stats['Completion'] / team_stats['Attempts']), 1)
team_stats['PctYardsPass'] = round(100 * (team_stats['Yards'] / team_stats['Total_Yards']), 1)
team_stats['PctYardsRush'] = round(100 * (team_stats['RushYards'] / team_stats['Total_Yards']), 1)
team_stats['SuccessRate4thDown'] = round(
    100 * (team_stats['Success4thDown'] / (team_stats['Success4thDown'] + team_stats['Failed4thDown'])), 1)
team_stats['SuccessRate3rdDown'] = round(
    100 * (team_stats['Success3rdDown'] / (team_stats['Success3rdDown'] + team_stats['Failed3rdDown'])), 1)
team_stats['PossessionTime'] = (60 * team_stats['TimeMinutes']) + team_stats['TimeSeconds']

team_stats[['DayofWeek', 'Month', 'DayofMonth', 'Year']] = team_stats.date.str.split(' ', expand=True)
team_stats['DayofMonth'] = team_stats['DayofMonth'].str.rstrip(',')

game_months = [
    (team_stats['Month'] == 'Sep'),
    (team_stats['Month'] == 'Oct'),
    (team_stats['Month'] == 'Nov'),
    (team_stats['Month'] == 'Dec'),
    (team_stats['Month'] == 'Jan'),
    (team_stats['Month'] == 'Feb')
]
month_choices = [9, 10, 11, 12, 1, 2]

team_stats['Month'] = np.select(game_months, month_choices, default=False)
team_stats['DayofMonth'] = team_stats['DayofMonth'].astype(int)
team_stats['Year'] = team_stats['Year'].astype(int)

date_mapping = {
    'Year':'year',
    'DayofMonth':'day',
    'Month':'month'}

team_stats = team_stats.rename(mapper=date_mapping, axis=1)
team_stats['gamedate'] = pd.to_datetime(team_stats[['year', 'month', 'day']])

team_stats = team_stats.drop(columns={'date','Completion','Success4thDown','Failed4thDown','Success3rdDown','Failed3rdDown', 'day'})

### Offensive Player Statistics

In [68]:
player_offense = pd.read_csv('player_offense.csv')

new_offense_cols = ['randnumber','playername','team','completions','passattempts',
                    'passingyards','passingtds','interceptions','sackstaken','yardslostbysack',
                    'longestcompletion','passrating','rushingattempts','rushyards','rushtds',
                    'longestrush','targets','receptions','receivingyards','receivingtds',
                    'longestreception','fumbles','fumbleslost','gamedate']

player_offense.columns = new_offense_cols

realplayer = player_offense['playername'] != "Player"
playerhere = player_offense['playername'].notna()
player_offense = player_offense[realplayer]
player_offense = player_offense[playerhere]

player_offense[['dayofweek','month','day','year']] = player_offense.gamedate.str.split(' ', expand=True)
player_offense = player_offense.drop(columns='gamedate')
player_offense['day'] = player_offense['day'].str.rstrip(',')

game_months = [
    (player_offense['month'] == 'Sep'),
    (player_offense['month'] == 'Oct'),
    (player_offense['month'] == 'Nov'),
    (player_offense['month'] == 'Dec'),
    (player_offense['month'] == 'Jan'),
    (player_offense['month'] == 'Feb')
]

player_offense['month'] = np.select(game_months, month_choices, default=False)
player_offense['gamedate'] = pd.to_datetime(player_offense[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in player_offense.columns if col not in skip_cols]
player_offense[cols_to_change] = player_offense[cols_to_change].astype(float)

player_offense['completionpct'] = round(100 * (player_offense['completions'] / player_offense['passattempts']), 1)
player_offense['td2int'] = round(player_offense['passingtds'] / player_offense['interceptions'], 3)
player_offense['yardsperrush'] = round(player_offense['rushyards'] / player_offense['rushingattempts'], 1)

player_offense = player_offense.drop(columns={'randnumber','day','month','year'})

/var/folders/xv/_79n8tfd6mx4j2z8m6rstygr0000gn/T/ipykernel_4137/3713386421.py:14: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  player_offense = player_offense[playerhere]


In [69]:
qbs = player_offense[player_offense['passattempts'] >= 10]
rbs = player_offense[(player_offense['rushingattempts'] >= 10) & (player_offense['passattempts'] <= 2)]
wrs = player_offense[player_offense['targets'] >= 2]

### Defensive Player Statistics

In [70]:
player_defense = pd.read_csv('player_defense.csv')

new_defense_cols = ['randnum','playername','team','interceptions','intrtnyards','int2td','intlongestyards',
                    'passdeflections','sacks','soloassisttackles','solotackles','assisttackles','tackelsforloss',
                    'qbhits','fumblerecovery','fumblereturnyards','fumble2td','forcedfumble','gamedate']
player_defense.columns = new_defense_cols

player_defense = player_defense[(player_defense['playername'] != 'Player') & player_defense['playername'].notna()]

player_defense[['dayofweek','month','day','year']] = player_defense.gamedate.str.split(' ', expand=True)
player_defense = player_defense.drop(columns='gamedate')
player_defense['day'] = player_defense['day'].str.rstrip(',')

game_months = [
    (player_defense['month'] == 'Sep'),
    (player_defense['month'] == 'Oct'),
    (player_defense['month'] == 'Nov'),
    (player_defense['month'] == 'Dec'),
    (player_defense['month'] == 'Jan'),
    (player_defense['month'] == 'Feb')
]

player_defense['month'] = np.select(game_months, month_choices, default=False)
player_defense['gamedate'] = pd.to_datetime(player_defense[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in player_defense.columns if col not in skip_cols]
player_defense[cols_to_change] = player_defense[cols_to_change].astype(float)

player_defense = player_defense.drop(columns={'randnum','day','month','year'})

### Advanced Passing Statistics

In [71]:
advanced_passing = pd.read_csv('passing_advanced.csv')
advanced_passing = advanced_passing.drop(columns={'Unnamed: 0','Cmp','1D'})
new_advpass_cols = ['playername','team','attempts','passingyards','firstdownsperpassplay',
                    'intendairyards','intendairyardsperpassatt','completedairyards',
                    'completedairyardspercomp','completedairyardsperpassatt',
                    'yardsaftercatch','yacpercompletion','drops','percentdrops',
                    'badthrows','badthrowspercent','sackstaken','timesblitzed',
                    'timeshurried','hitstaken','timespressured','percentdbspressured',
                    'scrambles','yardsperscramble','date']

advanced_passing.columns = new_advpass_cols

advanced_passing = advanced_passing[advanced_passing['playername'] != 'Player']

advanced_passing['percentdbspressured'] = advanced_passing['percentdbspressured'].str.rstrip('%')
advanced_passing['badthrowspercent'] = advanced_passing['badthrowspercent'].str.rstrip('%')
advanced_passing['percentdrops'] = advanced_passing['percentdrops'].str.rstrip('%')

advanced_passing[['dayofweek','month','day','year']] = advanced_passing.date.str.split(' ', expand=True)
advanced_passing = advanced_passing.drop(columns='date')
advanced_passing['day'] = advanced_passing['day'].str.rstrip(',')

game_months = [
    (advanced_passing['month'] == 'Sep'),
    (advanced_passing['month'] == 'Oct'),
    (advanced_passing['month'] == 'Nov'),
    (advanced_passing['month'] == 'Dec'),
    (advanced_passing['month'] == 'Jan'),
    (advanced_passing['month'] == 'Feb')
]

advanced_passing['month'] = np.select(game_months, month_choices, default=False)
advanced_passing['gamedate'] = pd.to_datetime(advanced_passing[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in advanced_passing.columns if col not in skip_cols]
advanced_passing[cols_to_change] = advanced_passing[cols_to_change].astype(float)

advanced_passing = advanced_passing.drop(columns={'day','month','year'})

### Advanced Receiving Statistics

In [72]:
advanced_receiving = pd.read_csv('receiving_advanced.csv')
advanced_receiving = advanced_receiving.drop(columns={'Unnamed: 0','Rec/Br'})
new_advrec_cols = ['playername','team','targets','receptions','yards','touchdowns',
                   'firstdownsreceiving','yardsbeforecatch','yardsbeforecatchper',
                   'yardsaftercatch','yardsaftercatchper','avgdeptoftarget',
                   'brokentackles','drops','droprate','intsontargets',
                   'passrtgontargets','date']

advanced_receiving.columns = new_advrec_cols

advanced_receiving = advanced_receiving[advanced_receiving['playername'] != 'Player']

advanced_receiving[['dayofweek','month','day','year']] = advanced_receiving.date.str.split(' ', expand=True)
advanced_receiving = advanced_receiving.drop(columns='date')
advanced_receiving['day'] = advanced_receiving['day'].str.rstrip(',')

game_months = [
    (advanced_receiving['month'] == 'Sep'),
    (advanced_receiving['month'] == 'Oct'),
    (advanced_receiving['month'] == 'Nov'),
    (advanced_receiving['month'] == 'Dec'),
    (advanced_receiving['month'] == 'Jan'),
    (advanced_receiving['month'] == 'Feb')
]

advanced_receiving['month'] = np.select(game_months, month_choices, default=False)
advanced_receiving['gamedate'] = pd.to_datetime(advanced_receiving[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in advanced_receiving.columns if col not in skip_cols]
advanced_receiving[cols_to_change] = advanced_receiving[cols_to_change].astype(float)

advanced_receiving = advanced_receiving.drop(columns={'day','month','year'})

### Advanced Rushing

In [73]:
advanced_rushing = pd.read_csv('rushing_advanced.csv')
advanced_rushing = advanced_rushing.drop(columns={'Unnamed: 0'})
new_advrush_cols = ['playername','team','rushes','attempts','touchdowns','firstdownsrushing',
                    'yardsbeforecontact','yardsbeforecontactper','yardsaftercontact',
                    'yardsaftercontactper','brokentackles','brokentacklesper','date']

advanced_rushing.columns = new_advrush_cols

advanced_rushing = advanced_rushing[advanced_rushing['playername'] != 'Player']

advanced_rushing[['dayofweek','month','day','year']] = advanced_rushing.date.str.split(' ', expand=True)
advanced_rushing = advanced_rushing.drop(columns='date')
advanced_rushing['day'] = advanced_rushing['day'].str.rstrip(',')

game_months = [
    (advanced_rushing['month'] == 'Sep'),
    (advanced_rushing['month'] == 'Oct'),
    (advanced_rushing['month'] == 'Nov'),
    (advanced_rushing['month'] == 'Dec'),
    (advanced_rushing['month'] == 'Jan'),
    (advanced_rushing['month'] == 'Feb')
]

advanced_rushing['month'] = np.select(game_months, month_choices, default=False)
advanced_rushing['gamedate'] = pd.to_datetime(advanced_rushing[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in advanced_rushing.columns if col not in skip_cols]
advanced_rushing[cols_to_change] = advanced_rushing[cols_to_change].astype(float)

advanced_rushing = advanced_rushing.drop(columns={'day','month','year'})

### Advanced Defensive Statistics

In [74]:
advanced_defense = pd.read_csv('defense_advanced.csv')
new_advdef_cols = ['randnum','playername','team','interceptions','deftargets','completions',
                   'cmppctincoverage','yardsallowed','yardsallowedpercmp',
                   'yardspertarget','tdsallowed','passrtgallowed',
                   'agvdepthoftargetcvg','airyardsoncmps','yardsaftercatch',
                   'blitzes','qbhurries','qbknockdown','sacks','pressures',
                   'soloassisttackles','missedtackles','missedtacklepct','date']
advanced_defense.columns = new_advdef_cols

advanced_defense = advanced_defense.drop(columns={'randnum', 'completions','airyardsoncmps'})

advanced_defense = advanced_defense[advanced_defense['playername'] != 'Player']

advanced_defense['cmppctincoverage'] = advanced_defense['cmppctincoverage'].str.rstrip('%')
advanced_defense['missedtacklepct'] = advanced_defense['missedtacklepct'].str.rstrip('%')

advanced_defense[['dayofweek','month','day','year']] = advanced_defense.date.str.split(' ', expand=True)
advanced_defense = advanced_defense.drop(columns='date')
advanced_defense['day'] = advanced_defense['day'].str.rstrip(',')

game_months = [
    (advanced_defense['month'] == 'Sep'),
    (advanced_defense['month'] == 'Oct'),
    (advanced_defense['month'] == 'Nov'),
    (advanced_defense['month'] == 'Dec'),
    (advanced_defense['month'] == 'Jan'),
    (advanced_defense['month'] == 'Feb')
]

advanced_defense['month'] = np.select(game_months, month_choices, default=False)
advanced_defense['gamedate'] = pd.to_datetime(advanced_defense[['year','month','day']])

skip_cols = ['playername','team','dayofweek','gamedate']
cols_to_change = [col for col in advanced_defense.columns if col not in skip_cols]
advanced_defense[cols_to_change] = advanced_defense[cols_to_change].astype(float)

advanced_defense = advanced_defense.drop(columns={'day','month','year'})

## Export to CSV 1
Ensuring that I don't have to rerun all these chunks of code each time

In [75]:
print(f'Current: {os.getcwd()}')
os.chdir('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/cleaned')
print(f'Current: {os.getcwd()}')

Current: /Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/htmlparse
Current: /Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/cleaned


In [ ]:
#team_stats.to_csv('team_stats.csv')
#qbs.to_csv('qbs.csv')
#rbs.to_csv('rbs.csv')
#wrs.to_csv('wrs.csv')
#player_defense.to_csv('player_defense.csv')
#advanced_passing.to_csv('adv_pass.csv')
#advanced_receiving.to_csv('adv_receiving.csv')
#advanced_rushing.to_csv('adv_rush.csv')
#advanced_defense.to_csv('adv_def.csv')

## Pre-Analysis Cleaning

In [2]:
print(f'Current: {os.getcwd()}')
os.chdir('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/cleaned')
print(f'Current: {os.getcwd()}')

Current: /Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL
Current: /Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/cleaned


### Team Stats

In [3]:
team_stats = pd.read_csv('team_stats.csv')
team_stats = team_stats.drop(columns={'Unnamed: 0','TimeMinutes','TimeSeconds'})
team_stats['Yards_Per_Penalty'] = round(team_stats['PenaltyYards'] / team_stats['Penalties'], 1)

# Seeing some irregularities with the 2025 data, I want to rescrap the data
# Maybe look at the code I wrote to initially scrape for issues
team_stats = team_stats[team_stats['year'] != 2025]

avg_stats = team_stats.groupby(
    ['team_name','year']).agg(
        {'First_Downs':'mean', 'Turnovers':'mean', 'Attempts':'mean', 'Yards':'mean',
         'Interceptions':'mean', 'RushAttempts':'mean', 'RushYards':'mean',
         'CompletionPct':'mean', 'PctYardsPass':'mean', 'PctYardsRush':'mean',
         'SuccessRate4thDown':'mean', 'SuccessRate3rdDown':'mean', 'PossessionTime':'mean',
         'Yards_Per_Penalty':'mean', 'Penalties':'mean'
            }).round(1)

avg_stats = avg_stats.add_suffix('_avg')

team_stats_b2g = pd.merge(left=team_stats, right=avg_stats, left_on=['team_name','year'], right_on=['team_name','year'], how='left')

team_stats_b2g = team_stats_b2g[['team_name'] + [col for col in team_stats_b2g.columns if col != 'team_name']]
team_stats_b2g = team_stats_b2g[['gamedate'] + [col for col in team_stats_b2g.columns if col != 'gamedate']]
team_stats_b2g = team_stats_b2g.drop(columns={'DayofWeek','month'})

team_stats_b2g.columns = team_stats_b2g.columns.str.lower()

In [79]:
team_stats_b2g

,gamedate,team_name,first_downs,net_pass_yards,total_yards,turnovers,attempts,yards,touchdowns,interceptions,...,rushattempts_avg,rushyards_avg,completionpct_avg,pctyardspass_avg,pctyardsrush_avg,successrate4thdown_avg,successrate3rddown_avg,possessiontime_avg,yards_per_penalty_avg,penalties_avg
0,2022-11-06,GNB,19,283,389,3,43,291,1,3,...,26.3,122.8,66.3,70.9,33.8,22.9,27.2,1929.9,8.2,4.9
1,2022-11-06,DET,19,137,254,1,26,137,2,1,...,27.3,117.9,64.7,70.5,31.9,32.8,29.2,1757.8,8.2,5.0
2,2024-12-22,PHI,20,127,338,2,28,154,1,1,...,35.3,176.2,67.2,57.3,48.8,34.3,27.4,1912.1,7.9,6.1
3,2024-12-22,WAS,23,255,368,5,39,258,5,2,...,30.9,150.4,70.7,65.0,39.7,44.0,29.7,1866.2,8.6,6.5
4,2023-11-19,DAL,23,204,311,0,41,204,2,0,...,27.5,108.3,66.6,73.4,31.3,31.9,31.1,1869.8,8.2,6.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2215,2021-11-08,PIT,20,175,280,1,30,205,2,0,...,23.3,87.6,64.6,77.6,27.2,21.7,28.1,1752.9,7.8,6.3
2216,2023-12-24,DAL,19,242,339,1,32,253,2,0,...,27.5,108.3,66.6,73.4,31.3,31.9,31.1,1869.8,8.2,6.7
2217,2023-12-24,MIA,22,284,375,0,37,293,1,0,...,27.1,131.1,68.2,71.1,32.8,22.8,27.8,1856.9,7.8,6.0
2218,2024-11-14,WAS,18,171,264,1,32,191,1,1,...,30.9,150.4,70.7,65.0,39.7,44.0,29.7,1866.2,8.6,6.5


### Player Offense

#### QBs

In [4]:
qbs = pd.read_csv('qbs.csv')

qbs = qbs.drop(columns={
    'Unnamed: 0','targets','receptions','receivingyards','receivingtds',
    'longestreception','dayofweek'})

###
# Changing values that initially came out as infinity with a missing value in stead
# This gets rid of the values that were dividing by zero
###
qbs['td2int'] = qbs['td2int'].replace(np.inf, np.nan)

avg_qb_stats = qbs.groupby('playername').agg(
    {'passattempts':'mean', 'passingyards':'mean', 'passingtds':'mean',
     'interceptions':'mean', 'sackstaken':'mean', 'yardslostbysack':'mean',
     'passrating':'mean', 'rushingattempts':'mean', 'rushtds':'mean',
     'completionpct':'mean'}).round(1)

avg_qb_stats = avg_qb_stats.add_suffix('_avg')

qb_stats_b2g = pd.merge(left=qbs, right=avg_qb_stats, left_on='playername', right_on='playername', how='left')

qb_stats_b2g = qb_stats_b2g[['gamedate'] + [col for col in qb_stats_b2g.columns if col != 'gamedate']]


In [81]:
qb_stats_b2g

,gamedate,playername,team,completions,passattempts,passingyards,passingtds,interceptions,sackstaken,yardslostbysack,...,passattempts_avg,passingyards_avg,passingtds_avg,interceptions_avg,sackstaken_avg,yardslostbysack_avg,passrating_avg,rushingattempts_avg,rushtds_avg,completionpct_avg
0,2022-11-06,Aaron Rodgers,GNB,23.0,43.0,291.0,1.0,3.0,1.0,8.0,...,33.1,234.0,1.8,0.5,2.1,15.2,98.9,1.7,0.1,66.0
1,2022-11-06,Jared Goff,DET,14.0,26.0,137.0,2.0,1.0,0.0,0.0,...,34.4,261.4,1.7,0.6,1.9,13.4,100.9,1.8,0.0,68.7
2,2024-12-22,Kenny Pickett,PHI,14.0,24.0,143.0,1.0,1.0,3.0,25.0,...,28.9,183.1,0.6,0.5,2.0,14.3,81.7,3.9,0.2,62.6
3,2024-12-22,Jayden Daniels,WAS,24.0,39.0,258.0,5.0,2.0,1.0,3.0,...,31.2,230.7,1.6,0.5,2.7,13.9,98.7,9.5,0.4,68.6
4,2023-11-19,Dak Prescott,DAL,25.0,38.0,189.0,2.0,0.0,0.0,0.0,...,35.8,262.6,2.0,0.8,2.1,12.3,98.7,3.2,0.1,68.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2363,2025-01-04,Russell Wilson,PIT,17.0,31.0,148.0,1.0,0.0,4.0,29.0,...,30.3,222.5,1.5,0.5,3.0,20.2,96.1,4.0,0.2,63.9
2364,2023-12-24,Dak Prescott,DAL,20.0,32.0,253.0,2.0,0.0,4.0,11.0,...,35.8,262.6,2.0,0.8,2.1,12.3,98.7,3.2,0.1,68.1
2365,2023-12-24,Tua Tagovailoa,MIA,24.0,37.0,293.0,1.0,0.0,1.0,9.0,...,33.0,257.0,1.7,0.7,1.7,11.8,98.0,2.2,0.1,68.4
2366,2024-11-14,Jayden Daniels,WAS,22.0,32.0,191.0,1.0,1.0,3.0,20.0,...,31.2,230.7,1.6,0.5,2.7,13.9,98.7,9.5,0.4,68.6


#### RBs

In [5]:
rbs = pd.read_csv('rbs.csv')

rbs = rbs.drop(columns={
    'Unnamed: 0','completions','passingyards','passingtds','interceptions',
    'sackstaken','yardslostbysack','longestcompletion','passrating',
    'longestreception','dayofweek','completionpct','td2int','passattempts'})

avg_rb_stats = rbs.groupby('playername').agg(
    {'rushingattempts':'mean', 'rushyards':'mean', 'targets':'mean',
     'receptions':'mean', 'receivingyards':'mean', 'fumbles':'mean',
     'yardsperrush':'mean'}).round(1)

avg_rb_stats = avg_rb_stats.add_suffix('_avg')

rb_stats_b2g = pd.merge(left=rbs, right=avg_rb_stats, left_on='playername', right_on='playername', how='left')

col_2_move = rb_stats_b2g.pop('gamedate')

rb_stats_b2g.insert(0, 'gamedate', col_2_move)

In [83]:
rb_stats_b2g

,gamedate,playername,team,rushingattempts,rushyards,rushtds,longestrush,targets,receptions,receivingyards,...,fumbles,fumbleslost,yardsperrush,rushingattempts_avg,rushyards_avg,targets_avg,receptions_avg,receivingyards_avg,fumbles_avg,yardsperrush_avg
0,2022-11-06,AJ Dillon,GNB,11.0,34.0,0.0,9.0,4.0,2.0,10.0,...,1.0,0.0,3.1,14.2,55.2,2.4,1.9,16.9,0.0,3.9
1,2022-11-06,Jamaal Williams,DET,24.0,81.0,0.0,14.0,0.0,0.0,0.0,...,0.0,0.0,3.4,15.1,58.8,1.3,1.1,6.7,0.2,3.8
2,2024-12-22,Saquon Barkley,PHI,29.0,150.0,2.0,68.0,1.0,0.0,0.0,...,0.0,0.0,5.2,18.9,89.7,4.1,3.0,19.7,0.1,4.7
3,2024-12-22,Brian Robinson Jr.,WAS,10.0,24.0,0.0,6.0,3.0,2.0,17.0,...,2.0,2.0,2.4,15.9,66.1,1.9,1.5,12.1,0.2,4.2
4,2023-11-19,Tony Pollard,DAL,12.0,61.0,1.0,22.0,5.0,4.0,19.0,...,0.0,0.0,5.1,15.6,71.8,3.6,2.9,19.0,0.1,4.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2265,2025-01-04,Najee Harris,PIT,12.0,36.0,1.0,11.0,5.0,4.0,20.0,...,0.0,0.0,3.0,16.8,66.0,3.4,2.6,16.8,0.1,3.8
2266,2023-12-24,Tony Pollard,DAL,12.0,38.0,0.0,7.0,1.0,1.0,5.0,...,0.0,0.0,3.2,15.6,71.8,3.6,2.9,19.0,0.1,4.7
2267,2023-12-24,Raheem Mostert,MIA,11.0,46.0,0.0,15.0,1.0,1.0,4.0,...,0.0,0.0,4.2,14.4,72.5,2.0,1.6,11.2,0.2,5.0
2268,2024-11-14,Brian Robinson Jr.,WAS,16.0,63.0,1.0,18.0,1.0,1.0,9.0,...,0.0,0.0,3.9,15.9,66.1,1.9,1.5,12.1,0.2,4.2


#### WRs (TEs)

In [6]:
wrs = pd.read_csv('wrs.csv')

wrs = wrs.drop(columns={
    'Unnamed: 0','completions','passattempts','passingyards','interceptions','sackstaken','yardslostbysack',
    'longestcompletion','passrating','rushingattempts','rushyards','rushtds','longestrush','fumbles',
    'fumbleslost','dayofweek','completionpct','td2int','yardsperrush','passingtds'})

avg_wr_stats = wrs.groupby('playername').agg({
    'targets':'mean', 'longestreception':'mean',
    'receptions':'mean', 'receivingyards':'mean',
    'receivingtds':'mean'}).round(1)

avg_wr_stats = avg_wr_stats.add_suffix('_avg')

wr_stats_b2g = pd.merge(left=wrs, right=avg_wr_stats, left_on='playername', right_on='playername', how='left')

col_2_move = wr_stats_b2g.pop('gamedate')

wr_stats_b2g.insert(0, 'gamedate', col_2_move)

In [130]:
wr_stats_b2g

,gamedate,playername,team,targets,receptions,receivingyards,receivingtds,longestreception,targets_avg,longestreception_avg,receptions_avg,receivingyards_avg,receivingtds_avg
0,2022-11-06,AJ Dillon,GNB,4.0,2.0,10.0,0.0,7.0,3.4,13.9,2.7,23.5,0.1
1,2022-11-06,Aaron Jones,GNB,2.0,2.0,20.0,0.0,15.0,4.4,13.9,3.5,26.1,0.2
2,2022-11-06,Allen Lazard,GNB,10.0,4.0,87.0,1.0,47.0,5.5,21.2,3.3,43.0,0.4
3,2022-11-06,Josiah Deguara,GNB,5.0,5.0,41.0,0.0,25.0,3.1,16.1,2.5,24.3,0.1
4,2022-11-06,Samori Toure,GNB,4.0,2.0,34.0,0.0,32.0,3.0,13.8,1.1,15.0,0.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13797,2024-11-14,Saquon Barkley,PHI,3.0,2.0,52.0,0.0,43.0,4.4,13.6,3.3,22.7,0.1
13798,2024-11-14,A.J. Brown,PHI,8.0,5.0,65.0,0.0,25.0,8.3,29.4,5.3,79.7,0.5
13799,2024-11-14,Dallas Goedert,PHI,5.0,5.0,61.0,0.0,32.0,5.8,21.0,4.4,53.3,0.3
13800,2024-11-14,DeVonta Smith,PHI,6.0,4.0,29.0,0.0,21.0,7.0,26.2,4.9,64.0,0.4


### Advanced Offensive Metrics

#### Passing

In [7]:
adv_passing = pd.read_csv('adv_pass.csv')

adv_passing = adv_passing.drop(columns={
    'Unnamed: 0','attempts','intendairyardsperpassatt',
    'completedairyardspercomp','completedairyardsperpassatt','drops',
    'percentdrops','hitstaken','dayofweek'})

adv_passing['pctyardsyac'] = round(100 * (adv_passing['yardsaftercatch'] / adv_passing['passingyards']),1)

avg_adv_pass = adv_passing.groupby('playername').agg({
    'yardsaftercatch':'mean', 'badthrows':'mean', 'badthrowspercent':'mean',
    'timesblitzed':'mean', 'timespressured':'mean', 'scrambles':'mean'}).round(1)

avg_adv_pass = avg_adv_pass.add_suffix('_avg')

adv_passing = adv_passing.drop(columns='passingyards')

adv_pass_stats_b2g = pd.merge(left=adv_passing, right=avg_adv_pass,
                              left_on='playername', right_on='playername', how='left')

full_passing = pd.merge(left=qb_stats_b2g, right=adv_pass_stats_b2g,
                        left_on=('playername','gamedate','team','sackstaken'),
                        right_on=('playername','gamedate','team','sackstaken'), how='left')

In [ ]:
full_passing

#### Rushing

In [8]:
adv_rushing = pd.read_csv('adv_rush.csv')

adv_rushing = adv_rushing.drop(columns={
    'Unnamed: 0','rushes','attempts','touchdowns'})

avg_adv_rush = adv_rushing.groupby('playername').agg({
    'yardsbeforecontact':'mean', 'yardsaftercontact':'mean'}).round(1)

avg_adv_rush = avg_adv_rush.add_suffix('_avg')

adv_rush_stats_b2g = pd.merge(left=adv_rushing, right=avg_adv_rush,
                              left_on='playername', right_on='playername', how='left')

full_rushing = pd.merge(left=rb_stats_b2g, right=adv_rush_stats_b2g,
                        left_on=('gamedate','playername','team'),
                        right_on=('gamedate','playername','team'),
                        how='left')

In [ ]:
full_rushing

#### Recieving

In [9]:
adv_receiving = pd.read_csv('adv_receiving.csv')

adv_receiving = adv_receiving.drop(columns={
    'Unnamed: 0','targets','receptions','yards','touchdowns','brokentackles',
    'intsontargets'})

avg_adv_rec = adv_receiving.groupby('playername').agg({
    'yardsbeforecatch':'mean', 'yardsaftercatch':'mean',
    'avgdeptoftarget':'mean', 'passrtgontargets':'mean'}).round(1)

avg_adv_rec = avg_adv_rec.add_suffix('_avg')

adv_rec_stats_b2g = pd.merge(left=adv_receiving, right=avg_adv_rec,
                             left_on='playername',right_on='playername',
                             how='left')

full_receiving = pd.merge(left=wr_stats_b2g, right=adv_rec_stats_b2g,
                          left_on=('playername','team','gamedate'),
                          right_on=('playername','team','gamedate'),
                          how='left')

In [141]:
full_receiving

,gamedate,playername,team,targets,receptions,receivingyards,receivingtds,longestreception,targets_avg,longestreception_avg,...,yardsaftercatchper,avgdeptoftarget,drops,droprate,passrtgontargets,dayofweek,yardsbeforecatch_avg,yardsaftercatch_avg,avgdeptoftarget_avg,passrtgontargets_avg
0,2022-11-06,AJ Dillon,GNB,4.0,2.0,10.0,0.0,7.0,3.4,13.9,...,3.0,3.5,0.0,0.0,56.2,Sunday,0.1,18.4,0.7,87.8
1,2022-11-06,Aaron Jones,GNB,2.0,2.0,20.0,0.0,15.0,4.4,13.9,...,12.5,-2.5,0.0,0.0,108.3,Sunday,-1.6,26.4,0.5,93.1
2,2022-11-06,Allen Lazard,GNB,10.0,4.0,87.0,1.0,47.0,5.5,21.2,...,7.5,15.0,0.0,0.0,65.4,Sunday,27.7,11.4,11.9,89.0
3,2022-11-06,Josiah Deguara,GNB,5.0,5.0,41.0,0.0,25.0,3.1,16.1,...,2.6,5.6,0.0,0.0,100.8,Sunday,4.1,11.0,3.1,89.0
4,2022-11-06,Samori Toure,GNB,4.0,2.0,34.0,0.0,32.0,3.0,13.8,...,4.0,24.3,0.0,0.0,79.2,Sunday,10.9,2.4,15.3,71.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13797,2024-11-14,Saquon Barkley,PHI,3.0,2.0,52.0,0.0,43.0,4.4,13.6,...,23.5,3.0,0.0,0.0,109.7,Thursday,-1.2,21.3,1.1,82.8
13798,2024-11-14,A.J. Brown,PHI,8.0,5.0,65.0,0.0,25.0,8.3,29.4,...,4.4,9.1,0.0,0.0,88.0,Thursday,53.2,25.5,12.4,98.7
13799,2024-11-14,Dallas Goedert,PHI,5.0,5.0,61.0,0.0,32.0,5.8,21.0,...,5.8,6.4,0.0,0.0,117.5,Thursday,24.7,26.8,7.0,101.3
13800,2024-11-14,DeVonta Smith,PHI,6.0,4.0,29.0,0.0,21.0,7.0,26.2,...,4.5,7.2,1.0,16.7,77.8,Thursday,42.7,21.2,11.2,101.9


## Export to CSV 2

In [10]:
print(f'Current: {os.getcwd()}')
os.chdir('/Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/finished')
print(f'Current: {os.getcwd()}')

Current: /Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/cleaned
Current: /Users/camsmithers/Desktop/Camalytics/CamalyticsEnv/Projects/Sports/NFL/Data/Datasets/finished


In [11]:
team_stats_b2g.to_csv('full_team_stats.csv')
full_passing.to_csv('full_passing.csv')
full_rushing.to_csv('full_rushing.csv')
full_receiving.to_csv('full_receiving.csv')